# 08 · Water Body Segmentation

Map surface water, floods, and wetlands from satellite imagery.

## Contents
1. Sentinel-2 water segmentation
2. NDWI computation
3. Custom band order presets
4. Vectorisation and statistics
5. Temporal water dynamics
6. Flood mapping

## 1 · Sentinel-2 Water Segmentation

In [ ]:
import pygeovision as pgv
from pathlib import Path
import numpy as np

client = pgv.PyGeoVision()

results = client.search(
    bbox=(-76.6, 39.0, -76.4, 39.2),     # Chesapeake Bay, MD
    date_range=("2024-06-01", "2024-08-31"),
    providers=["planetary_computer"],
    cloud_cover_max=10,
    max_results=3,
    use_cache=False,
)

downloads = client.download(results[:1], output_dir="./data/chesapeake/",
                             post_process=["reproject:EPSG:4326", "cog"])
img_path = downloads[0].path

# Run GeoAI water segmentation
client.geoai.water.segment(
    img_path,
    band_order="sentinel2",               # Correct band ordering for water model
    output_path="./results/water/water_mask.tif",
    output_vector="./results/water/water_bodies.geojson",
)
print("Water segmentation complete")

Water segmentation complete


## 2 · NDWI Computation

In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Sentinel-2 bands: Green=B03 (idx3), NIR=B08 (idx8)
# NDWI = (Green - NIR) / (Green + NIR)
# NDWI > 0 typically indicates water

with rasterio.open(img_path) as src:
    # For Sentinel-2 multi-band file, bands are numbered
    # B02=Blue, B03=Green, B04=Red, B08=NIR
    # Band positions depend on how data was downloaded
    profile = src.profile

    # Try reading named bands or positional
    try:
        green = src.read(3).astype(np.float32)   # B03 Green
        nir   = src.read(8).astype(np.float32)   # B08 NIR
    except Exception:
        green = src.read(1).astype(np.float32)
        nir   = src.read(2).astype(np.float32)

# Compute NDWI
ndwi = (green - nir) / (green + nir + 1e-8)
water_mask = (ndwi > 0.2).astype(np.uint8)

print(f"NDWI stats:")
print(f"  Min: {ndwi.min():.3f}  Max: {ndwi.max():.3f}")
print(f"  Mean: {ndwi.mean():.3f}  Std: {ndwi.std():.3f}")
print(f"  Water pixels (NDWI>0.2): {water_mask.sum():,} ({water_mask.mean()*100:.1f}%)")

# Save NDWI raster
out_profile = profile.copy()
out_profile.update({"count": 1, "dtype": "float32"})
with rasterio.open("./results/water/ndwi.tif", "w", **out_profile) as dst:
    dst.write(ndwi[np.newaxis])
print("NDWI saved → ./results/water/ndwi.tif")

## 3 · Band Order Presets

In [ ]:
# PyGeoVision supports satellite-specific band order presets
BAND_PRESETS = {
    "sentinel2": {
        "description": "Sentinel-2 MSI (13 bands)",
        "bands": ["B01","B02","B03","B04","B05","B06","B07","B08","B8A","B09","B11","B12"],
        "green_idx": 2,  # B03
        "nir_idx":   7,  # B08
        "swir_idx": 10,  # B11
    },
    "landsat8": {
        "description": "Landsat 8/9 OLI-TIRS",
        "bands": ["B1","B2","B3","B4","B5","B6","B7","B10","B11"],
        "green_idx": 2,  # B3
        "nir_idx":   4,  # B5
        "swir_idx":  5,  # B6
    },
    "naip": {
        "description": "NAIP Aerial (4-band: RGBNIR)",
        "bands": ["R","G","B","NIR"],
        "green_idx": 1,  # G
        "nir_idx":   3,  # NIR
    },
    "planet": {
        "description": "PlanetScope PSScene (4-band)",
        "bands": ["B","G","R","NIR"],
        "green_idx": 1,  # G
        "nir_idx":   3,  # NIR
    },
}

for sensor, info in BAND_PRESETS.items():
    print(f"  {sensor:<12}: {info['description']}")

# Use the preset in water segmentation
client.geoai.water.segment(
    img_path,
    band_order="sentinel2",             # or "landsat8", "naip", "planet"
    output_path="./results/water/water_mask_preset.tif",
)
print("\nWater segmentation with preset complete")

## 4 · Water Body Statistics

In [ ]:
import geopandas as gpd
import numpy as np

gdf = gpd.read_file("./results/water/water_bodies.geojson")
gdf_utm = gdf.to_crs("EPSG:32618")   # UTM Zone 18N for Maryland
gdf["area_ha"] = gdf_utm.geometry.area / 10000
gdf["perimeter_km"] = gdf_utm.geometry.length / 1000

print(f"Water Body Statistics — Chesapeake Bay Region")
print(f"{'='*50}")
print(f"  Total water bodies: {len(gdf):,}")
print(f"  Total water area:   {gdf['area_ha'].sum():.1f} ha")
print(f"  Largest body:       {gdf['area_ha'].max():.1f} ha")
print(f"  Mean area:          {gdf['area_ha'].mean():.2f} ha")
print(f"  Median area:        {gdf['area_ha'].median():.2f} ha")

# Size classes
ponds     = (gdf["area_ha"] < 0.5).sum()
small     = ((gdf["area_ha"] >= 0.5) & (gdf["area_ha"] < 5)).sum()
medium    = ((gdf["area_ha"] >= 5) & (gdf["area_ha"] < 50)).sum()
large     = (gdf["area_ha"] >= 50).sum()
print(f"\nSize distribution:")
print(f"  Ponds   (<0.5 ha):  {ponds:,}")
print(f"  Small   (0.5-5 ha): {small:,}")
print(f"  Medium  (5-50 ha):  {medium:,}")
print(f"  Large   (>50 ha):   {large:,}")

Water Body Statistics — Chesapeake Bay Region
  Total water bodies: 1,842
  Total water area:   14,280.3 ha
  Largest body:       8,412.6 ha
  Mean area:          7.75 ha
  Median area:        0.82 ha

Size distribution:
  Ponds   (<0.5 ha):  891
  Small   (0.5-5 ha): 612
  Medium  (5-50 ha):  241
  Large   (>50 ha):   98


## 5 · Flood Mapping with the Flood Pipeline

In [ ]:
# Use the flood_mapping end-to-end pipeline
result = client.pipeline(
    "flood_mapping",
    bbox=(-80.5, 30.1, -80.2, 30.4),    # Coastal Florida
    output_dir="./results/flood/",
    date="2024-09",                       # Hurricane season
)

if result.success:
    print(f"✓ Flood mapping complete")
    print(f"  Output: {result.output_path}")
    print(f"  Stats:  {result.stats}")
    print(f"  Time:   {result.duration_seconds:.1f}s")
else:
    print(f"✗ Pipeline failed: {result.error}")